# Credict — Module 1: User Acquisition & KYC Funnel
## Part 1 of 3: Data Generation + Acquisition Metrics

**What we cover here:**
- Generate `devices` and `sessions` table data
- Daily and Monthly New Users
- Acquisition Channel Mix
- City Tier Breakdown
- Month-over-Month Growth

---

## Step 1: Imports + DB Connection

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
from sqlalchemy import create_engine
from faker import Faker
from datetime import date, timedelta, datetime
import random
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('en_IN')
Faker.seed(SEED)

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
})
PALETTE = ['#534AB7','#1D9E75','#D85A30','#BA7517','#D4537E','#378ADD']

# ── DB Connection — update credentials ───────────────────────
PG_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'database': 'credict',
    'user':     'postgres',
    'password': 'your_password',
}
DB_URL = (
    f"postgresql+psycopg2://{PG_CONFIG['user']}:{PG_CONFIG['password']}"
    f"@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['database']}"
)
engine = create_engine(DB_URL, echo=False)
conn   = psycopg2.connect(**PG_CONFIG)
conn.autocommit = True
cur    = conn.cursor()

cur.execute('SELECT COUNT(*) FROM users')
print(f'✅ Connected | Users in DB: {cur.fetchone()[0]:,}')

---
## Step 2: Generate Devices Table Data

Each user has 1 primary device. ~8% have 2 devices (upgraded phone or tablet).  
Device type mix: 78% Android, 22% iOS — realistic for Indian fintech apps.

In [ ]:
print('🔄 Generating devices...')

df_users = pd.read_sql(
    'SELECT user_id, signup_date FROM users ORDER BY user_id',
    engine
)

ANDROID_MODELS = [
    'Samsung Galaxy M34','Samsung Galaxy A54','Redmi Note 12',
    'Redmi Note 13','POCO M6 Pro','Realme Narzo 60',
    'Vivo V29e','OnePlus Nord CE3','OPPO Reno 11'
]
IOS_MODELS = [
    'iPhone 13','iPhone 14','iPhone 14 Plus',
    'iPhone 15','iPhone 12'
]
ANDROID_OS = ['13','13','14','14','12','12']
IOS_OS     = ['16','16','17','17','15']
APP_VERSIONS = ['2.1.0','2.2.0','2.3.0','2.3.1','2.4.0']
NETWORKS   = ['4g','4g','4g','5g','wifi','3g']

device_rows = []
dev_id = 1

for _, user in df_users.iterrows():
    uid        = int(user['user_id'])
    signup_dt  = datetime.combine(user['signup_date'], datetime.min.time())
    is_android = random.random() < 0.78

    if is_android:
        d_type = 'android'
        model  = random.choice(ANDROID_MODELS)
        os_ver = random.choice(ANDROID_OS)
    else:
        d_type = 'ios'
        model  = random.choice(IOS_MODELS)
        os_ver = random.choice(IOS_OS)

    app_ver    = random.choice(APP_VERSIONS)
    first_seen = signup_dt + timedelta(minutes=random.randint(0, 30))
    last_seen  = first_seen + timedelta(days=random.randint(1, 300))

    device_rows.append((
        dev_id, uid, d_type, model, os_ver,
        app_ver, first_seen, last_seen, True
    ))
    dev_id += 1

    # ~8% have a second device
    if random.random() < 0.08:
        is_android2 = random.random() < 0.78
        d2_type = 'android' if is_android2 else 'ios'
        model2  = random.choice(ANDROID_MODELS if is_android2 else IOS_MODELS)
        os2     = random.choice(ANDROID_OS if is_android2 else IOS_OS)
        fs2     = first_seen + timedelta(days=random.randint(30, 180))
        ls2     = fs2 + timedelta(days=random.randint(1, 150))
        device_rows.append((
            dev_id, uid, d2_type, model2, os2,
            random.choice(APP_VERSIONS), fs2, ls2, False
        ))
        dev_id += 1

dev_sql = '''
    INSERT INTO devices
        (device_id, user_id, device_type, device_model,
         os_version, app_version, first_seen_at, last_seen_at, is_primary)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''
CHUNK = 2000
for i in range(0, len(device_rows), CHUNK):
    cur.executemany(dev_sql, device_rows[i:i+CHUNK])

cur.execute('SELECT COUNT(*) FROM devices')
print(f'✅ Devices inserted: {cur.fetchone()[0]:,}')

cur.execute("SELECT device_type, COUNT(*) FROM devices GROUP BY device_type")
for row in cur.fetchall():
    print(f'   {row[0]}: {row[1]:,}')

---
## Step 3: Generate Sessions Table Data

Sessions are the unit of engagement. Rules:
- KYC-pending users: 1–3 sessions (mostly onboarding)
- KYC-approved users: higher session frequency (they use the app for payments/loans)
- Sessions span the user's life from signup to Dec 2024
- Session duration: 3–25 minutes, log-normal distribution

In [ ]:
print('🔄 Generating sessions...')

START_DATE = date(2024, 1, 1)
END_DATE   = date(2024, 12, 31)

# Fetch users with KYC status and their primary device
df_user_kyc = pd.read_sql('''
    SELECT
        u.user_id,
        u.signup_date,
        k.kyc_status,
        d.device_id,
        d.device_type
    FROM users u
    LEFT JOIN user_kyc k  ON u.user_id = k.user_id
    LEFT JOIN devices d   ON u.user_id = d.user_id
                         AND d.is_primary = TRUE
''', engine)

session_rows = []
sess_id = 1

for _, row in df_user_kyc.iterrows():
    uid       = int(row['user_id'])
    dev_id    = int(row['device_id']) if pd.notna(row['device_id']) else None
    signup_dt = row['signup_date']
    kyc_ok    = row['kyc_status'] == 'approved'

    # Days available for this user
    days_active = (END_DATE - signup_dt).days
    if days_active <= 0:
        continue

    # Session frequency per week:
    # KYC-approved → 2–8 sessions/week   (active fintech user)
    # KYC-pending  → 1–3 sessions total  (just onboarding)
    if kyc_ok:
        weeks       = days_active // 7
        total_sess  = sum(random.randint(2, 8) for _ in range(max(weeks, 1)))
    else:
        total_sess  = random.randint(1, 3)

    # Spread sessions randomly across the user's active window
    session_days = sorted(
        random.choices(range(days_active), k=total_sess)
    )

    for day_offset in session_days:
        sess_date   = signup_dt + timedelta(days=day_offset)
        hour        = random.choices(
            range(24),
            weights=[1,1,1,1,1,1,2,4,6,7,8,7,6,6,7,8,9,10,9,8,7,5,3,2],
            k=1
        )[0]
        minute      = random.randint(0, 59)
        sess_start  = datetime(sess_date.year, sess_date.month,
                               sess_date.day, hour, minute)
        # Log-normal duration: realistic app session lengths
        dur_sec     = int(np.clip(np.random.lognormal(5.5, 0.8), 60, 1800))
        sess_end    = sess_start + timedelta(seconds=dur_sec)
        screens     = max(1, dur_sec // 45)  # ~1 screen per 45 sec
        app_ver     = random.choice(['2.1.0','2.2.0','2.3.0','2.3.1','2.4.0'])
        network     = random.choice(['4g','4g','4g','5g','wifi','3g'])

        session_rows.append((
            sess_id, uid, dev_id,
            sess_start, sess_end, sess_date,
            dur_sec, screens, app_ver, network
        ))
        sess_id += 1

sess_sql = '''
    INSERT INTO sessions
        (session_id, user_id, device_id, session_start,
         session_end, session_date, duration_sec,
         screen_count, app_version, network_type)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''
CHUNK = 2000
for i in range(0, len(session_rows), CHUNK):
    cur.executemany(sess_sql, session_rows[i:i+CHUNK])

cur.execute('SELECT COUNT(*) FROM sessions')
print(f'✅ Sessions inserted: {cur.fetchone()[0]:,}')

cur.execute('SELECT MIN(session_date), MAX(session_date) FROM sessions')
row = cur.fetchone()
print(f'   Date range: {row[0]} → {row[1]}')

---
## Step 4: Acquisition Metrics — SQL Queries

### 4.1 — Daily New Users

In [ ]:
df_daily = pd.read_sql('''
    SELECT
        signup_date,
        COUNT(user_id)                                 AS new_users,
        SUM(COUNT(user_id)) OVER (ORDER BY signup_date) AS cumulative_users
    FROM users
    GROUP BY signup_date
    ORDER BY signup_date
''', engine, parse_dates=['signup_date'])

print(f'📊 Daily new users — {len(df_daily)} days')
print(f'   Avg per day : {df_daily.new_users.mean():.0f}')
print(f'   Peak day    : {df_daily.signup_date[df_daily.new_users.idxmax()].date()} ({df_daily.new_users.max():,} users)')
print(f'   Total users : {df_daily.cumulative_users.iloc[-1]:,}')
df_daily.head(5)

### 4.2 — Monthly New Users + MoM Growth

In [ ]:
df_monthly = pd.read_sql('''
    WITH monthly AS (
        SELECT
            TO_CHAR(signup_date, \'YYYY-MM\')          AS month,
            COUNT(user_id)                            AS new_users
        FROM users
        GROUP BY 1
    )
    SELECT
        month,
        new_users,
        LAG(new_users) OVER (ORDER BY month)         AS prev_month_users,
        ROUND(
            100.0 * (new_users - LAG(new_users) OVER (ORDER BY month))
            / NULLIF(LAG(new_users) OVER (ORDER BY month), 0)
        , 1)                                         AS mom_growth_pct
    FROM monthly
    ORDER BY month
''', engine)

print('📊 Monthly new users with MoM growth:')
print(df_monthly.to_string(index=False))

### 4.3 — Channel Mix

In [ ]:
df_channel = pd.read_sql('''
    SELECT
        TO_CHAR(signup_date, \'YYYY-MM\')              AS month,
        signup_channel,
        COUNT(user_id)                                AS new_users,
        ROUND(
            100.0 * COUNT(user_id)
            / SUM(COUNT(user_id)) OVER (
                PARTITION BY TO_CHAR(signup_date, \'YYYY-MM\')
            ), 1
        )                                             AS channel_share_pct
    FROM users
    GROUP BY 1, 2
    ORDER BY 1, 2
''', engine)

# Pivot for readability
df_channel_pivot = df_channel.pivot(index='month', columns='signup_channel', values='new_users').fillna(0).astype(int)
print('📊 New users by channel per month:')
print(df_channel_pivot.to_string())

### 4.4 — City Tier Breakdown

In [ ]:
df_tier = pd.read_sql('''
    SELECT
        city_tier,
        COUNT(user_id)                                        AS total_users,
        ROUND(100.0 * COUNT(user_id) / SUM(COUNT(user_id)) OVER (), 1)
                                                              AS share_pct,
        ROUND(100.0 * COUNT(CASE WHEN plan_type = \'premium\' THEN 1 END)
              / NULLIF(COUNT(user_id), 0), 1)                 AS premium_rate_pct
    FROM users
    GROUP BY city_tier
    ORDER BY city_tier
''', engine)

print('📊 Users by city tier:')
print(df_tier.to_string(index=False))

---
## Step 5: Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Credict — Module 1 | User Acquisition Overview', fontsize=15, fontweight='bold', y=1.01)

# ── Chart 1: Daily New Users + Cumulative ─────────────────────
ax = axes[0, 0]
ax2_twin = ax.twinx()
ax.bar(df_daily['signup_date'], df_daily['new_users'],
       color=PALETTE[0], alpha=0.6, label='Daily New Users')
ax2_twin.plot(df_daily['signup_date'], df_daily['cumulative_users'],
              color=PALETTE[1], linewidth=2.5, label='Cumulative')
ax.set_title('Daily Signups + Cumulative Growth')
ax.set_xlabel('Date')
ax.set_ylabel('Daily New Users', color=PALETTE[0])
ax2_twin.set_ylabel('Cumulative Users', color=PALETTE[1])
ax2_twin.spines['right'].set_visible(True)
ax2_twin.spines['top'].set_visible(False)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax2_twin.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x/1000)}K'))
lines1, lbl1 = ax.get_legend_handles_labels()
lines2, lbl2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1+lines2, lbl1+lbl2, loc='upper left', fontsize=9)

# ── Chart 2: Monthly New Users + MoM Growth ───────────────────
ax = axes[0, 1]
df_m = df_monthly.copy()
bars = ax.bar(df_m['month'], df_m['new_users'],
              color=PALETTE[0], alpha=0.85, edgecolor='white')
for bar, row in zip(bars, df_m.itertuples()):
    if pd.notna(row.mom_growth_pct):
        clr   = '#1D9E75' if row.mom_growth_pct >= 0 else '#D85A30'
        arrow = '▲' if row.mom_growth_pct >= 0 else '▼'
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 30,
            f'{arrow}{abs(row.mom_growth_pct):.0f}%',
            ha='center', va='bottom', fontsize=8, color=clr, fontweight='bold'
        )
ax.set_title('Monthly New Users (MoM Growth %)')
ax.set_xlabel('Month')
ax.set_ylabel('New Users')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# ── Chart 3: Channel Mix — Stacked Bar ───────────────────────
ax = axes[1, 0]
df_pct = df_channel.pivot(
    index='month', columns='signup_channel', values='channel_share_pct'
).fillna(0)
channels = df_pct.columns.tolist()
bottom   = np.zeros(len(df_pct))
for i, ch in enumerate(channels):
    ax.bar(df_pct.index, df_pct[ch], bottom=bottom,
           label=ch, color=PALETTE[i % len(PALETTE)], alpha=0.85, edgecolor='white')
    bottom += df_pct[ch].values
ax.set_title('Acquisition Channel Mix % (Monthly)')
ax.set_xlabel('Month')
ax.set_ylabel('Share (%)')
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# ── Chart 4: City Tier Breakdown ─────────────────────────────
ax = axes[1, 1]
tier_colors = {'tier1': PALETTE[0], 'tier2': PALETTE[1], 'tier3': PALETTE[2]}
bars = ax.bar(
    df_tier['city_tier'],
    df_tier['total_users'],
    color=[tier_colors[t] for t in df_tier['city_tier']],
    alpha=0.85, edgecolor='white', width=0.5
)
for bar, row in zip(bars, df_tier.itertuples()):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 200,
        f'{row.total_users:,}\n({row.share_pct}%)',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() / 2,
        f'Premium:\n{row.premium_rate_pct}%',
        ha='center', va='center', fontsize=9, color='white', fontweight='bold'
    )
ax.set_title('Users by City Tier')
ax.set_xlabel('City Tier')
ax.set_ylabel('Total Users')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.set_ylim(0, df_tier['total_users'].max() * 1.2)

plt.tight_layout()
plt.savefig('../assets/charts/01_acquisition_part1.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Chart saved → assets/charts/01_acquisition_part1.png')

---
## Step 6: Key Insights — Part 1

Write your observations here after running the queries. Structure: **Observation → So What → Recommendation**

### Insight 1 — Growth Curve
**Observation:** User signups follow an accelerating curve (beta distribution), with H2 2024 contributing ~60% of total signups.

**So What:** Growth is not linear — the product is gaining distribution momentum. This is typical of a referral-driven fintech app where network effects compound over time.

**Recommendation:** Double down on the referral channel in Q4 — it delivers the lowest CAC and benefits most from the growing user base acting as referrers.

---

### Insight 2 — Channel Mix
**Observation:** Organic (30%) + Referral (25%) together account for 55% of acquisition. Paid channels (Google + Meta) = 35%.

**So What:** More than half of growth is low-cost or zero-cost. This is a strong signal that the product has genuine word-of-mouth traction.

**Recommendation:** Track KYC completion and LTV separately by channel — referral users typically have higher activation rates than paid users. If confirmed, shift budget from paid to referral incentives.

---

### Insight 3 — City Tier Distribution
**Observation:** Tier 1 cities (40%) have a higher premium plan conversion rate (~12%) vs Tier 2 (6%) and Tier 3 (~6%).

**So What:** Tier 1 users have higher income, higher credit worthiness, and are more willing to pay for premium features. However, Tier 2/3 together represent 60% of users and are undermonetised.

**Recommendation:** For Tier 2/3, build a ₹99/month plan with features tuned to their needs (loan top-up alerts, EMI reminders) — separate from the Tier 1 premium offering.

---

**→ Part 2: KYC Funnel Analysis**